# Schema

In [1]:
import pandas as pd

# View 1 — Core message info
messages_core = pd.DataFrame([
    {"msg_id": "m001", "label": "phishing", "subject": "Verify Account", "body_text": "Click this link...", "provenance_tag": "phishpot"},
    {"msg_id": "m002", "label": "benign", "subject": "Meeting Agenda", "body_text": "Please find attached...", "provenance_tag": "enron"},
])

# View 2 — Header + Auth features
headers_auth = pd.DataFrame([
    {"msg_id": "m001", "spf_result": "fail", "dkim_result": "none", "num_received": 5, "return_path": "noreply@bank-login.com"},
    {"msg_id": "m002", "spf_result": "pass", "dkim_result": "pass", "num_received": 3, "return_path": "user@enron.com"},
])

# View 3 — URL + Attachment features
urls_files = pd.DataFrame([
    {"msg_id": "m001", "url_list": ["http://bank-login.com"], "attach_exts": [], "url_entropy": 4.5, "malware_hash_hits": 0},
    {"msg_id": "m002", "url_list": [], "attach_exts": ["pdf"], "url_entropy": None, "malware_hash_hits": 0},
])


In [2]:
import pandas as pd
import numpy as np

# For headers
headers_auth["has_spf"] = headers_auth["spf_result"].notna().astype(int)
headers_auth["has_dkim"] = headers_auth["dkim_result"].notna().astype(int)
headers_auth["has_headers"] = (
    headers_auth[["spf_result", "dkim_result", "num_received", "return_path"]]
    .notna()
    .any(axis=1)
    .astype(int)
)

# For URLs/attachments
urls_files["has_url"] = urls_files["url_list"].apply(lambda x: int(bool(x)))
urls_files["has_attachment"] = urls_files["attach_exts"].apply(lambda x: int(bool(x)))
urls_files["has_threatintel"] = (
    (urls_files["has_url"] | urls_files["has_attachment"]).astype(int)
)


In [3]:
merged = (
    messages_core
    .merge(headers_auth, on="msg_id", how="left")
    .merge(urls_files, on="msg_id", how="left")
)


In [4]:
merged

,msg_id,label,subject,body_text,provenance_tag,spf_result,dkim_result,num_received,return_path,has_spf,has_dkim,has_headers,url_list,attach_exts,url_entropy,malware_hash_hits,has_url,has_attachment,has_threatintel
0,m001,phishing,Verify Account,Click this link...,phishpot,fail,none,5,noreply@bank-login.com,1,1,1,[http://bank-login.com],[],4.5,0,1,0,1
1,m002,benign,Meeting Agenda,Please find attached...,enron,pass,pass,3,user@enron.com,1,1,1,[],[pdf],NaN,0,0,1,1
